# HANDS-ON EXERCISE 3: BANKING CUSTOMER TRANSACTION ANALYSIS

## Introduction

This exercise focuses on analysing a banking customer transaction dataset using Pandas. The dataset contains information on customers, their locations, transaction types, transaction amounts, and account statuses. The objective is to inspect the raw transaction data, identify and investigate data-quality problems, apply appropriate cleaning techniques, and generate useful business insights from the cleaned dataset.

The analysis will involve examining the structure and quality of the dataset, investigating missing and unexpected values in the transaction amount variable, creating a transaction classification based on transaction amount, and performing descriptive and grouped analyses. The results will be interpreted from a business perspective to demonstrate how data analysis can support understanding of customer transaction patterns and financial activity.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Banking_Customer_Transation.CSV")

In [4]:
df.head()

,Customer ID,Customer Name,City,Transaction Type,Amount,Account Status
0,CUST00585,Tunde Okafor,Kaduna,Online Transfer,37974.75,Active
1,CUST01061,Bola Uche,Lagos,Deposit,9896.89,Active
2,CUST00077,Kelechi Nwosu,Abuja,Transfer,59302.75,Active
3,CUST00898,Tunde Abubakar,Benin City,Deposit,71719.97,Dormant
4,CUST00594,Amina Ojo,Lagos,Online Transfer,3979.55,Active


In [5]:
df.shape

(3000, 6)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Customer ID       3000 non-null   object
 1   Customer Name     2990 non-null   object
 2   City              2976 non-null   object
 3   Transaction Type  2982 non-null   object
 4   Amount            2884 non-null   object
 5   Account Status    2988 non-null   object
dtypes: object(6)
memory usage: 140.8+ KB


### Data Structure and Quality Findings

The dataset contains 3,000 transaction records across six columns. The inspection shows that all six variables are currently stored as object data types. Missing values are present in Customer Name, City, Transaction Type, Amount, and Account Status, with Amount having the highest number of missing observations at 116. The Amount column requires particular attention because transaction amounts should be numerical, but the variable is currently stored as text. Further investigation is therefore required before converting or analysing the transaction amounts.

In [8]:
df.describe()

,Customer ID,Customer Name,City,Transaction Type,Amount,Account Status
count,3000,2990,2976,2982,2884,2988
unique,1031,400,10,7,2803,4
top,CUST01098,Femi Ekanem,Lagos,Transfer,Not Available,Active
freq,9,16,731,548,20,2371


### Initial Descriptive Summary

The descriptive summary confirms that the variables are being interpreted as categorical or text data rather than numerical data. The Amount column contains 2,884 non-missing observations and has seven distinct values, indicating that unexpected text values may be present alongside transaction amounts. Therefore, numerical summary statistics such as the mean, minimum, and maximum cannot yet be obtained for the Amount variable. Further investigation of the individual values is required before deciding how the column should be cleaned.

### Part A - Initial Inspection Summary

The banking transaction dataset contains 3,000 records and six variables: Customer ID, Customer Name, City, Transaction Type, Amount, and Account Status. The initial inspection shows that all six variables are stored as object data types. Missing values are present in Customer Name, City, Transaction Type, Amount, and Account Status, with the Amount column containing 116 missing observations. The descriptive summary also shows that Amount is being treated as a categorical variable rather than a numerical variable, indicating that it requires further investigation before transaction values can be analysed numerically.

In [9]:
df["Amount"].unique()

array(['37974.75', '9896.89', '59302.75', ..., '8349.08', '288077.4',
       '20047.09'], shape=(2804,), dtype=object)

In [10]:
df["Amount"].value_counts().head(20)

Amount
Not Available    20
Error            19
Pending          16
Unknown           9
1O000             8
##VALUE!          6
0                 5
-2500             4
122682.31         2
₦12,500           2
6871.71           1
86046.06          1
25482.96          1
9219.06           1
91926.43          1
37974.75          1
52346.24          1
36019.44          1
165568.89         1
54109.07          1
Name: count, dtype: int64

In [12]:
df["Amount"].isnull().sum()

np.int64(116)

In [13]:
pd.to_numeric(df["Amount"], errors="coerce").isnull().sum()

np.int64(196)

In [14]:
df.loc[
    pd.to_numeric(df["Amount"], errors="coerce").isnull(),
    "Amount"
    ].value_counts()

Amount
Not Available    20
Error            19
Pending          16
Unknown           9
1O000             8
##VALUE!          6
₦12,500           2
Name: count, dtype: int64

### Investigation of Transaction Amount

The Amount variable was investigated for missing and unexpected values before cleaning. There were 116 missing observations and 80 non-numeric entries. Among the non-numeric entries, `₦12,500` represents a monetary value stored with currency formatting, while `1O000` appears to be a typographical error in which the letter O was used instead of zero. Other entries, including `Not Available`, `Error`, `Pending`, `Unknown`, and `##VALUE!`, do not provide a reliable numerical transaction amount and therefore require appropriate treatment before analysis.

In [39]:
amount_numeric = pd.to_numeric(df["Amount"], errors="coerce")

df.loc[amount_numeric.isna() & df["Amount"].notna(), "Amount"].value_counts()

Series([], Name: count, dtype: int64)

In [40]:
df["Amount"].map(type).value_counts()

Amount
<class 'float'>    3000
Name: count, dtype: int64

In [41]:
df = pd.read_csv("Banking_Customer_Transation.csv")

In [42]:
df["Amount"].map(type).value_counts()

Amount
<class 'str'>      2884
<class 'float'>     116
Name: count, dtype: int64

In [43]:
df["Amount"].value_counts().head(20)

Amount
Not Available    20
Error            19
Pending          16
Unknown           9
1O000             8
##VALUE!          6
0                 5
-2500             4
122682.31         2
₦12,500           2
6871.71           1
86046.06          1
25482.96          1
9219.06           1
91926.43          1
37974.75          1
52346.24          1
36019.44          1
165568.89         1
54109.07          1
Name: count, dtype: int64

### Part B - Investigation of Transaction Amount

The `Amount` column was investigated for missing values, unexpected text, and potentially invalid numerical entries. The investigation identified 116 missing values and 80 non-numeric text entries. Among the text entries, some values such as `1O000` and `₦12,500` appear to be recoverable formatting or entry errors, while values such as `Not Available`, `Error`, `Pending`, `Unknown`, and `##VALUE!` do not provide a reliable transaction amount and will require appropriate treatment during the cleaning stage.

The values `0` and `-2500` were identified as numerical but unusual transaction amounts. These values will not be removed automatically because their validity cannot be determined from the `Amount` column alone. They will first be examined alongside the associated transaction records before a cleaning decision is made.

In [44]:
df[df["Amount"].isin(["0", "-2500"])]

,Customer ID,Customer Name,City,Transaction Type,Amount,Account Status
161,CUST00659,Femi Ojo,Abuja,Transfer,0,Active
275,CUST00121,Musa Ibrahim,Lagos,Transfer,0,Active
1263,CUST00632,Kelechi Musa,Kano,Transfer,0,Active
1278,CUST00788,Kelechi Ekanem,Enugu,Online Transfer,-2500,Active
1369,CUST00379,Maryam Balogun,Ibadan,Deposit,0,Active
1559,CUST00467,Seyi Uche,Abuja,Deposit,-2500,Active
1654,CUST00959,Maryam Okeke,Lagos,POS Purchase,-2500,Active
2556,CUST00489,Amaka Onyeka,Port Harcourt,Online Transfer,0,Active
2819,CUST00506,Femi Ojo,Abuja,ATM Withdrawal,-2500,Dormant


### Investigation of Unusual Transaction Amounts

The investigation identified five transactions with an amount of zero and four transactions with an amount of -2,500. These records contain valid customer information, transaction types, cities, and account statuses. Although the values are unusual, the dataset does not provide sufficient information to determine whether they represent errors, reversals, refunds, or other legitimate transaction adjustments. Therefore, the values will be retained rather than deleted or altered during the cleaning process.

In [47]:
df[df["Amount"].isin(["0", "-2500"])]

,Customer ID,Customer Name,City,Transaction Type,Amount,Account Status
161,CUST00659,Femi Ojo,Abuja,Transfer,0,Active
275,CUST00121,Musa Ibrahim,Lagos,Transfer,0,Active
1263,CUST00632,Kelechi Musa,Kano,Transfer,0,Active
1278,CUST00788,Kelechi Ekanem,Enugu,Online Transfer,-2500,Active
1369,CUST00379,Maryam Balogun,Ibadan,Deposit,0,Active
1559,CUST00467,Seyi Uche,Abuja,Deposit,-2500,Active
1654,CUST00959,Maryam Okeke,Lagos,POS Purchase,-2500,Active
2556,CUST00489,Amaka Onyeka,Port Harcourt,Online Transfer,0,Active
2819,CUST00506,Femi Ojo,Abuja,ATM Withdrawal,-2500,Dormant


In [45]:
df["Amount"] = df["Amount"].replace({
    "1O000": "10000",
    "₦12,500": "12500"
})

In [46]:
df["Amount"].value_counts().loc[["10000", "12500"]]

Amount
10000    8
12500    2
Name: count, dtype: int64

### Handling Unrecoverable Text Values

Several entries in the `Amount` column contain text that does not represent a reliable numerical transaction amount. These include `Not Available`, `Error`, `Pending`, `Unknown`, and `##VALUE!`. Because no valid transaction amount can be inferred from these entries, they will be converted to missing values rather than being assigned arbitrary amounts. This preserves the integrity of the financial data and allows the missing observations to be handled appropriately during subsequent analysis.

In [48]:
df["Amount"] = df["Amount"].replace({
    "Not Available": pd.NA,
    "Error": pd.NA,
    "Pending": pd.NA,
    "Unknown": pd.NA,
    "##VALUE!": pd.NA
})

In [50]:
df["Amount"].isin(
    ["Not Available", "Error", "Pending", "Unknown", "##VALUE!"]
).sum()

np.int64(0)

In [51]:
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

In [52]:
df["Amount"].dtype

dtype('float64')

In [53]:
df["Amount"].isnull().sum()

np.int64(186)

### Part C - Cleaning Summary

The `Amount` column was cleaned by correcting recoverable data-entry errors and converting unrecoverable text values to missing values. The entry `1O000` was corrected to `10000`, while `₦12,500` was converted to `12500`. Values such as `Not Available`, `Error`, `Pending`, `Unknown`, and `##VALUE!` were treated as missing because no reliable numerical amount could be determined from them.

After cleaning, the `Amount` column was successfully converted from an object data type to a numerical (`float64`) data type. The dataset now contains 186 missing transaction amounts, comprising the original 116 missing values and 70 unrecoverable text entries. The zero and negative transaction values were retained because the available data did not provide sufficient evidence to classify them as invalid.

### Creating Transaction Categories

The cleaned transaction amounts will be classified using the specified business rule. Transactions with an amount of 100,000 or more will be classified as **Large Transaction**, while transactions below 100,000 will be classified as **Regular Transaction**. Records with missing transaction amounts will remain missing in the new classification column because their transaction category cannot be determined.

In [62]:
df["Transaction_Category"] = pd.cut(
    df["Amount"],
    bins=[-float("inf"), 99999.99, float("inf")],
    labels=["Regular Transaction", "Large Transaction"]
)

In [63]:
df["Transaction_Category"].value_counts(dropna=False)

Transaction_Category
Regular Transaction    2538
Large Transaction       276
NaN                     186
Name: count, dtype: int64

### Transaction Classification Result

The transaction amounts were successfully classified according to the specified business rule. A total of 276 transactions were classified as **Large Transactions**, while 2,538 were classified as **Regular Transactions**. The remaining 186 records could not be classified because their transaction amounts were missing. The classification therefore preserves the missing information rather than assigning an unsupported category.

### Number of Transactions by City

The number of transaction records will be examined across cities to identify the geographic distribution of the bank's transaction activity.

In [64]:
df["City"].value_counts(dropna=False)

City
Lagos            731
Abuja            495
Port Harcourt    353
Ibadan           314
Kano             302
Enugu            193
Kaduna           184
Benin City       165
Aba              130
Jos              109
NaN               24
Name: count, dtype: int64

### Transactions by City - Interpretation

The distribution of transactions varies considerably across the cities. Lagos recorded the highest number of transactions with 731 records, followed by Abuja with 495 and Port Harcourt with 353. Aba recorded the lowest number of identified city transactions with 130 records, while 24 records had missing city information. This distribution suggests that Lagos represents the largest transaction market in the dataset and may therefore be an important location for further business analysis.

### Average Transaction Amount

The overall average transaction amount will be calculated using the cleaned numerical `Amount` column. Records with missing transaction amounts will be excluded automatically from the mean calculation because no reliable transaction value is available for those records.

In [65]:
df["Amount"].mean()

np.float64(45076.65529140014)

### Average Transaction Amount - Interpretation

The average transaction amount in the cleaned dataset is approximately **₦45,076.66**. This value represents the mean transaction amount among records with available and usable transaction values. The 186 records with missing transaction amounts were excluded from the calculation because their transaction values could not be reliably determined during data cleaning.

In [66]:
(df["Transaction_Category"] == "Large Transaction").sum()

np.int64(276)

### Number of Large Transactions - Interpretation

A total of **276 transactions** were classified as Large Transactions because their transaction amounts were equal to or greater than ₦100,000. This represents a smaller proportion of the transactions with available amounts, while the majority were classified as Regular Transactions.

### Most Frequent Transaction Type

The frequency of each transaction type will be examined to identify the transaction method that occurs most often in the dataset.

In [67]:
df["Transaction Type"].value_counts()

Transaction Type
Transfer           548
Deposit            498
Withdrawal         465
Payment            406
ATM Withdrawal     377
POS Purchase       374
Online Transfer    314
Name: count, dtype: int64

### Most Frequent Transaction Type - Interpretation

**Transfer** was the most frequently recorded transaction type, with 548 transactions. Deposit followed with 498 transactions, while Online Transfer was the least frequent with 314 transactions. This suggests that conventional transfers represent an important component of transaction activity in the dataset.

### Average Transaction Amount by Transaction Type

The average transaction amount will be calculated for each transaction type using `groupby()`. This will help identify which transaction methods are associated with higher or lower average transaction values.

In [68]:
df.groupby("Transaction Type")["Amount"].mean()

Transaction Type
ATM Withdrawal     36960.128418
Deposit            44458.179765
Online Transfer    46682.385322
POS Purchase       50549.251497
Payment            40820.957268
Transfer           46319.721292
Withdrawal         48961.158676
Name: Amount, dtype: float64

### Average Transaction Amount by Transaction Type - Interpretation

The average transaction amount varied across transaction types. POS Purchase recorded the highest average transaction amount at approximately **₦50,549.25**, followed by Withdrawal at **₦48,961.16**. ATM Withdrawal had the lowest average transaction amount at approximately **₦36,960.13**. This variation suggests that customers may use different transaction channels for transactions of different monetary values, which could be useful for understanding customer transaction behaviour and service-channel preferences.

### Challenge - Average Transaction Amount by City

The average transaction amount will be calculated for each city using `groupby()`. This analysis will help identify whether transaction values differ across locations and determine the city with the highest average transaction amount.

In [69]:
df.groupby("City")["Amount"].mean().sort_values(ascending=False)

City
Benin City       52154.374228
Aba              50332.422314
Lagos            46905.360656
Kaduna           46313.136114
Abuja            46284.952060
Jos              44224.250000
Kano             43659.663636
Ibadan           42323.277869
Port Harcourt    41283.747620
Enugu            40047.124586
Name: Amount, dtype: float64

### Challenge - Business Interpretation

Benin City recorded the highest average transaction amount at approximately **₦52,154.37**, followed by Aba at **₦50,332.42** and Lagos at **₦46,905.36**. Enugu recorded the lowest average transaction amount among the cities with available transaction amounts, at approximately **₦40,047.12**. The differences in average transaction values may indicate variations in customer transaction behaviour, purchasing patterns, or the types of financial activities taking place across locations. From a business perspective, Benin City may warrant further investigation to understand the factors contributing to its relatively higher average transaction value.

In [71]:
df.groupby("Transaction_Category", observed=True)["Amount"].agg(
    Number_of_Transactions="count",
    Average_Transaction_Amount="mean"
)

,Number_of_Transactions,Average_Transaction_Amount
Transaction_Category,,
Regular Transaction,2538,30492.599992
Large Transaction,276,179186.555109


### Final Transaction Category Summary

The summary shows a clear difference between the two transaction categories. Regular Transactions accounted for **2,538 records** and had an average transaction amount of approximately **₦30,492.60**, while Large Transactions accounted for **276 records** with a much higher average of approximately **₦179,186.56**. This indicates that although Large Transactions occur less frequently, they involve substantially greater monetary values and may therefore be particularly important when assessing high-value customer activity and transaction risk.